# 🎵 Spotify Best of the Best - Data Analytics

**Capstone project for Coursera Google Data Analytics**

This notebook analyzes Spotify playlist data for Kaggle.

## 📊 Analysis Overview

This comprehensive Kaggle-ready analysis explores:
- Genre and language distributions
- Artist popularity across categories  
- Track duration patterns
- Correlations between musical attributes
- Text analysis of song titles
- Rare language discovery

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns
import re
from collections import Counter

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

### Load Dataset

**Note for Kaggle:** Update the file path to `/kaggle/input/your-dataset-name/playlist-data-fully-enriched.csv`

In [ ]:
# Load the dataset
# For Kaggle: Update this path to /kaggle/input/your-dataset-name/playlist-data-fully-enriched.csv
df = pd.read_csv('playlist-data-fully-enriched.csv')

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 2. Data Overview

In [ ]:
# Display basic information
print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Missing Values: {df.isnull().sum().sum():,}")
print(f"Duplicate Rows: {df.duplicated().sum():,}")
print("=" * 70)

In [ ]:
# Column information
column_info = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.astype(str).values,
    'Non-Null Count': df.count().values,
    'Null Count': df.isnull().sum().values,
    'Unique Values': [df[col].nunique() for col in df.columns]
})

print("\nCOLUMN INFORMATION:")
print(column_info.to_string(index=False))

In [ ]:
# Display first 10 rows
print("\nFIRST 10 ROWS:")
df.head(10)

---
## 3. Genre Analysis

In [ ]:
if 'genre' in df.columns:
    # Get genre counts
    genre_counts = df['genre'].value_counts()
    top_10_genres = genre_counts.head(10)
    
    # Create pie chart
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = plt.cm.Set3(range(len(top_10_genres)))
    wedges, texts, autotexts = ax.pie(
        top_10_genres.values, 
        labels=top_10_genres.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors
    )
    
    # Make percentage text more readable
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(10)
    
    for text in texts:
        text.set_fontsize(11)
    
    ax.set_title(f'Top 10 Music Genres Distribution (Total: {len(df):,} tracks)', 
                 fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"\n📊 GENRE STATISTICS:")
    print(f"Total Genres: {df['genre'].nunique()}")
    print(f"Most Common Genre: {genre_counts.index[0]}")
    print(f"Tracks in Most Common Genre: {genre_counts.values[0]:,} ({genre_counts.values[0]/len(df)*100:.1f}%)")
else:
    print("⚠️ No 'genre' column found in the dataset.")

---
## 4. Language Analysis

In [ ]:
if 'language' in df.columns:
    language_counts = df['language'].value_counts()
    top_10_languages = language_counts.head(10)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.Pastel1(range(len(top_10_languages)))
    
    def make_autopct(values):
        def my_autopct(pct):
            return f'{pct:.1f}%' if pct > 2 else ''
        return my_autopct
    
    wedges, texts, autotexts = ax.pie(
        top_10_languages.values, 
        autopct=make_autopct(top_10_languages.values),
        startangle=90,
        colors=colors,
        pctdistance=0.80,
        textprops={'fontsize': 12, 'weight': 'bold'}
    )
    
    for autotext in autotexts:
        autotext.set_color('black')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(13)
        autotext.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])
    
    total_tracks = top_10_languages.sum()
    legend_labels = [f'{lang}: {count:,} tracks ({count/total_tracks*100:.1f}%)' 
                   for lang, count in zip(top_10_languages.index, top_10_languages.values)]
    ax.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=11)
    
    ax.set_title(f'Top 10 Languages Distribution (Total: {len(df):,} tracks)', 
                 fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    print(f"\n🗣️ LANGUAGE STATISTICS:")
    print(f"Total Languages: {df['language'].nunique()}")
    print(f"Most Common Language: {language_counts.index[0]}")
    print(f"Tracks: {language_counts.values[0]:,} ({language_counts.values[0]/len(df)*100:.1f}%)")
else:
    print("⚠️ No 'language' column found.")

---
## 5. Genre Distribution by Language

In [ ]:
if 'genre' in df.columns and 'language' in df.columns:
    languages = ['English', 'Ukrainian', 'Polish']
    fig, axes = plt.subplots(3, 1, figsize=(14, 18))
    
    for idx, language in enumerate(languages):
        lang_df = df[df['language'] == language]
        
        if len(lang_df) > 0:
            genre_counts = lang_df['genre'].value_counts().head(10)
            ax = axes[idx]
            colors = plt.cm.Set3(range(len(genre_counts)))
            
            wedges, texts, autotexts = ax.pie(
                genre_counts.values,
                autopct='%1.1f%%',
                startangle=90,
                colors=colors,
                pctdistance=0.85
            )
            
            for autotext in autotexts:
                autotext.set_color('white')
                autotext.set_fontweight('bold')
                autotext.set_fontsize(11)
            
            legend_labels = [f'{genre}: {count:,} tracks ({count/len(lang_df)*100:.1f}%)' 
                           for genre, count in zip(genre_counts.index, genre_counts.values)]
            ax.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=10)
            ax.set_title(f'Genre Distribution - {language} (Total: {len(lang_df):,} tracks)', 
                        fontsize=14, fontweight='bold', pad=15)
            
            print(f"\n{language}: {len(lang_df):,} tracks, top genre: {genre_counts.index[0]}")
        else:
            axes[idx].text(0.5, 0.5, f'No data for {language}', ha='center', va='center')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Both 'genre' and 'language' columns required.")

---
## 6. Duration Analysis

In [ ]:
duration_col = None
if 'duration_minutes' in df.columns:
    duration_col = 'duration_minutes'
    duration_unit = 'minutes'
elif 'duration_ms' in df.columns:
    duration_col = 'duration_ms'
    duration_unit = 'milliseconds'

if duration_col:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(df[duration_col].dropna(), bins=30, edgecolor='black', color='skyblue')
    ax.set_xlabel(f'Duration ({duration_unit})', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Distribution of Track Duration', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\n⏱️ DURATION STATISTICS:")
    print(f"Mean: {df[duration_col].mean():.2f} {duration_unit}")
    print(f"Median: {df[duration_col].median():.2f} {duration_unit}")
    print(f"Min: {df[duration_col].min():.2f} {duration_unit}")
    print(f"Max: {df[duration_col].max():.2f} {duration_unit}")
else:
    print("⚠️ No duration column found.")

---
# 💡 Interesting Findings

## 7. Most Popular Elements

In [ ]:
print("=" * 70)
print("🏆 MOST POPULAR ELEMENTS")
print("=" * 70)

if 'genre' in df.columns:
    top_genre = df['genre'].value_counts().index[0]
    top_genre_count = df['genre'].value_counts().values[0]
    print(f"\n🎵 Most Popular Genre: {top_genre}")
    print(f"   Tracks: {top_genre_count:,} ({top_genre_count/len(df)*100:.1f}%)")

if 'language' in df.columns:
    top_language = df['language'].value_counts().index[0]
    top_language_count = df['language'].value_counts().values[0]
    print(f"\n🗣️ Most Popular Language: {top_language}")
    print(f"   Tracks: {top_language_count:,} ({top_language_count/len(df)*100:.1f}%)")

if 'artist' in df.columns:
    top_artist = df['artist'].value_counts().index[0]
    top_artist_count = df['artist'].value_counts().values[0]
    print(f"\n🎤 Most Popular Artist: {top_artist}")
    print(f"   Tracks: {top_artist_count:,}")

print("=" * 70)

## 8. Correlation Analysis

In [ ]:
if all(col in df.columns for col in ['artist', 'genre', 'language']):
    print("🔗 CORRELATION ANALYSIS")
    print("=" * 70)
    
    top_artist = df['artist'].value_counts().index[0]
    top_artist_df = df[df['artist'] == top_artist]
    
    if len(top_artist_df) > 0:
        artist_top_genre = top_artist_df['genre'].value_counts().index[0]
        artist_top_language = top_artist_df['language'].value_counts().index[0]
        
        top_genre = df['genre'].value_counts().index[0]
        top_language = df['language'].value_counts().index[0]
        
        print(f"\nTop Artist: {top_artist}")
        print(f"Primary Genre: {artist_top_genre}")
        print(f"Primary Language: {artist_top_language}")
        
        if artist_top_genre == top_genre:
            print(f"\n✅ The top artist performs in the most popular genre ({top_genre})")
        else:
            print(f"\n⚠️ Top artist's genre ({artist_top_genre}) differs from overall top genre ({top_genre})")
        
        if artist_top_language == top_language:
            print(f"✅ The top artist performs in the most popular language ({top_language})")
        else:
            print(f"⚠️ Top artist's language ({artist_top_language}) differs from overall top language ({top_language})")
    print("=" * 70)
else:
    print("⚠️ Required columns not found.")

## 9. Duration Extremes

In [ ]:
duration_col = 'duration_minutes' if 'duration_minutes' in df.columns else 'duration_ms' if 'duration_ms' in df.columns else None

if duration_col and all(col in df.columns for col in ['track_name', 'artist', 'language']):
    print("⏱️ DURATION EXTREMES")
    print("=" * 70)
    
    longest_idx = df[duration_col].idxmax()
    longest_song = df.loc[longest_idx]
    longest_min = longest_song[duration_col] / 60000 if duration_col == 'duration_ms' else longest_song[duration_col]
    
    print(f"\n🔼 LONGEST SONG:")
    print(f"   {longest_song['track_name']} - {longest_song['artist']}")
    print(f"   Duration: {longest_min:.2f} minutes")
    
    shortest_idx = df[duration_col].idxmin()
    shortest_song = df.loc[shortest_idx]
    shortest_min = shortest_song[duration_col] / 60000 if duration_col == 'duration_ms' else shortest_song[duration_col]
    
    print(f"\n🔽 SHORTEST SONG:")
    print(f"   {shortest_song['track_name']} - {shortest_song['artist']}")
    print(f"   Duration: {shortest_min:.2f} minutes")
    print("=" * 70)
else:
    print("⚠️ Required columns not found.")

## 10. Most Common Words in Song Titles

In [ ]:
if 'track_name' in df.columns:
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 
                 'of', 'with', 'by', 'from', 'is', 'it', 'as', 'be', 'was', 'are'}
    
    all_words = []
    for title in df['track_name'].dropna():
        words = re.findall(r'\b[a-zA-Z]{2,}\b', str(title).lower())
        all_words.extend([w for w in words if w not in stop_words])
    
    if all_words:
        word_counts = Counter(all_words)
        top_words = word_counts.most_common(10)
        
        print("📝 MOST COMMON WORDS IN SONG TITLES")
        print("=" * 70)
        for rank, (word, count) in enumerate(top_words, 1):
            print(f"{rank:2d}. {word:<15s} - {count:4d} occurrences ({count/len(df)*100:.2f}%)")
        print(f"\nTotal Unique Words: {len(word_counts):,}")
        print("=" * 70)
else:
    print("⚠️ 'track_name' column not found.")

## 11. Top Artists by Language

In [ ]:
if 'language' in df.columns and 'artist' in df.columns:
    print("🎤 TOP ARTISTS BY LANGUAGE (Top 5)")
    print("=" * 70)
    
    top_10_languages = df['language'].value_counts().head(10)
    
    for idx, (language, count) in enumerate(top_10_languages.items(), 1):
        print(f"\n{idx}. {language.upper()} ({count:,} tracks)")
        print("-" * 70)
        
        lang_artists = df[df['language'] == language]['artist'].value_counts().head(5)
        for rank, (artist, artist_count) in enumerate(lang_artists.items(), 1):
            percentage = (artist_count / count) * 100
            print(f"  {rank}. {artist:<30s} {artist_count:4d} tracks ({percentage:5.1f}%)")
    
    print("=" * 70)
else:
    print("⚠️ Required columns not found.")

## 12. Top Artists by Genre

In [ ]:
if 'genre' in df.columns and 'artist' in df.columns:
    print("🎸 TOP ARTISTS BY GENRE (Top 5)")
    print("=" * 70)
    
    top_10_genres = df['genre'].value_counts().head(10)
    
    for idx, (genre, count) in enumerate(top_10_genres.items(), 1):
        print(f"\n{idx}. {genre.upper()} ({count:,} tracks)")
        print("-" * 70)
        
        genre_artists = df[df['genre'] == genre]['artist'].value_counts().head(5)
        for rank, (artist, artist_count) in enumerate(genre_artists.items(), 1):
            percentage = (artist_count / count) * 100
            print(f"  {rank}. {artist:<30s} {artist_count:4d} tracks ({percentage:5.1f}%)")
    
    print("=" * 70)
else:
    print("⚠️ Required columns not found.")

## 13. Rarest Languages

In [ ]:
if all(col in df.columns for col in ['language', 'track_name', 'artist']):
    print("🌍 RAREST LANGUAGES - Tracks in the 10 Rarest Languages")
    print("=" * 70)
    
    language_counts = df['language'].value_counts()
    rarest_10_languages = language_counts.tail(10).sort_values()
    
    all_rare_tracks = []
    for language in rarest_10_languages.index:
        lang_tracks = df[df['language'] == language][['track_name', 'artist', 'language']].copy()
        all_rare_tracks.append(lang_tracks)
    
    if all_rare_tracks:
        combined_rare_tracks = pd.concat(all_rare_tracks, ignore_index=True)
        
        print(f"\nTotal tracks in rarest 10 languages: {len(combined_rare_tracks):,}")
        print(f"Rarest languages: {', '.join(rarest_10_languages.index.tolist())}\n")
        print("=" * 70)
        
        combined_rare_tracks.index = range(1, len(combined_rare_tracks) + 1)
        combined_rare_tracks.columns = ['Track Title', 'Artist', 'Language']
        print(combined_rare_tracks.to_string())
        print("=" * 70)
    else:
        print("No rare language tracks found")
else:
    print("⚠️ Required columns not found.")

---
## 🎯 Summary

This comprehensive analysis of the Spotify "Best of the Best" playlist revealed:

### Key Findings:
- **Genre Diversity**: Analysis of genre distribution and popularity patterns
- **Language Patterns**: Most common languages and their relationship with genres
- **Artist Insights**: Top artists across different languages and genres
- **Duration Characteristics**: Track length patterns and extremes
- **Text Analytics**: Common themes and words in track titles
- **Cultural Diversity**: Rare languages represented in the playlist

### Technical Stack:
- **Data Processing**: pandas, numpy
- **Visualization**: matplotlib, seaborn
- **Text Analysis**: regex, collections.Counter

---

**Project**: Capstone for Coursera Google Data Analytics  
**Author**: Oliwia Achyna  
**Date**: April 2026